# AI Multi-Agents, RAG, Routing, Guardrails, Observability, and Explainability with LangGraph, LangSmith, and Pydantic LogFire

In [1]:
import os
import streamlit as st
from dotenv import load_dotenv
from typing import TypedDict, Literal
from langchain_groq import ChatGroq
# from groq import Groq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END
import logfire
from langsmith import traceable
from langsmith import Client as LangSmithClient 

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# Activing the parallelism tokenization
os.environ['TOKENIZERS_PARALLELISM'] = 'True'
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    logfire.critical("GROQ_API_KEY was not defined in the environment or .env!") # Using logfire

In [3]:
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
langchain_api_key_env = os.getenv("LANGCHAIN_API_KEY")

LOGFIRE_API_KEY = os.getenv("LOGFIRE_API_KEY")

In [4]:
try:
    logfire.configure() 
    print("Log - Logfire setted.") 
except Exception as e:
     print(f"Log - Warning: Failed to configure Logfire automatically.: {e}")

Logfire project URL: https://logfire-us.pydantic.dev/krupck/starter-project

[transformers] Accessing `LambdaRuntimeClient` from `.models.aria.image_processing_aria`. Returning `LambdaRuntimeClient` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `LambdaRuntimeClient` from `.models.aria.image_processing_pil_aria`. Returning `LambdaRuntimeClient` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `LambdaRuntimeClient` from `.models.auto.image_processing_auto`. Returning `LambdaRuntimeClient` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `LambdaRuntimeClient` from `.models.beit.image_processing_beit`. Returning `LambdaRuntimeClient` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `LambdaRuntimeClient` from `.models.beit.image_processing_pil_beit`. Returning `LambdaRuntimeClient` instead. Behavior may be different and

Log - Logfire setted.


In [5]:
if not langsmith_api_key or not langchain_api_key_env:
    print("LANGSMITH_API_KEY and/or LANGCHAIN_API_KEY aren't defineds. LangSmith tracing may not working correctly.")

if not LOGFIRE_API_KEY:
    print("LOGFIRE_API_KEY not defined. Logs to Pydantic LogFire Cloud will not work (unless another OTEL exporter is configured).")

# Checking LANGCHAIN_TRACING_V2
if os.getenv("LANGCHAIN_TRACING_V2", "false").lower() != "true":
    print("The LANGCHAIN_TRACING_V2 environment variable is not set to 'true'. Automatic tracing from LangGraph to LangSmith MAY be disabled.")

# Checking LANGCHAIN_API_KEY specifically for tracing
if not langchain_api_key_env:
     print("The LANGCHAIN_API_KEY environment variable is not set. LangGraph tracing for LangSmith will NOT work.")

# RAG path
VECTORSTORE_PATH = "faiss_index"

The LANGCHAIN_TRACING_V2 environment variable is not set to 'true'. Automatic tracing from LangGraph to LangSmith MAY be disabled.


In [ ]:
def load_llm_final_answer():
    print("LOG - Loading LLM Groq...")
    
    try:
        llm = ChatGroq(api_key = groq_api_key, model = "openai/gpt-oss-120b", temperature = 0.1) 
        
        logfire.info("LLM Groq (resposta final) carregado com sucesso.")
        
        return llm
    
    except Exception as e:
        logfire.error("Error Loading final LLM", error = str(e), exc_info = True)
        

In [7]:
def load_retriever():
    print("LOG - Loading Retriever RAG...")
    
    if not os.path.exists(VECTORSTORE_PATH):
        logfire.error("FAISS index was not found", path = VECTORSTORE_PATH)
        
        print(f"FAISS was not found at '{VECTORSTORE_PATH}'. Run 'setup_rag.py'.")
    try:
        model_name = "BAAI/bge-base-en"
        encode_kwargs = {'normalize_embeddings': True} 
        embedding_model = HuggingFaceEmbeddings(
            model_name = model_name, 
            model_kwargs={'device': 'cpu'},
            encode_kwargs = encode_kwargs
        )
        
        vector_store = FAISS.load_local(VECTORSTORE_PATH, embedding_model, allow_dangerous_deserialization = True)
        retriever = vector_store.as_retriever(search_kwargs = {'k': 5})
        
        logfire.info("Retriever RAG loaded sucessfully.", path = VECTORSTORE_PATH)
        
        return retriever
    
    except Exception as e:
        logfire.error("Error loading Retriever RAG", path = VECTORSTORE_PATH, error = str(e), exc_info = True) 
        print(f"Erro loading Retriever RAG: {e}")

In [8]:
class GraphState(TypedDict):
    query: str
    source_decision: Literal["RAG", "WEB", ""]
    rag_context: str | None
    web_results: str | None
    final_answer: str | None